In [3]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import fastparquet
import os
import json
import gzip

# file_path = "../../data/amazon/initial_recs/recommendations.parquet"
# if os.path.exists(file_path):
#     df = pd.read_parquet(file_path, engine='fastparquet')
# else:
#     raise FileNotFoundError(f"Parquet file not found at: {file_path}")

In [4]:
# pf = pq.ParquetFile(file_path)
# print(pf.schema)
# print("First user's recommendations type:", type(df['recommendations'].iloc[0]))
# print("First recommendation item type:", type(df['recommendations'].iloc[0][0]))

In [5]:
def load_jsonl_gz(file_path):
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        for line in f:
            yield json.loads(line)

In [6]:
df = pd.DataFrame(list(load_jsonl_gz("recommendations.jsonl.gz")))

In [7]:
df

,user_id,recommendations
0,AE4QWKT4VX3DUVNTZM2OHJ2K7YMQ,"[{'iid': 'B000002U82', 'title': 'Dark Side Of ..."
1,AH3EHHDN2SCPSPE5FMS4MHUP5CEA,"[{'iid': 'B000B8QEZG', 'title': 'Confessions o..."
2,AEYOPYIIRMWVY5BLNZKZFZFJFQRA,"[{'iid': 'B00000J2PH', 'title': 'Carole King T..."
3,AHCSCMYFD5WLVXSYWSJGOJD65TMA,"[{'iid': 'B000002UB3', 'title': 'Abbey Road', ..."
4,AECJOIRAVDVSVD6OD5UZKW4PO2KQ,"[{'iid': 'B004EBT5CU', 'title': 'Oscd Ad Adele..."
...,...,...
57292,AGNSSC2S3TL7HZETBWSC6NDILSMA,"[{'iid': 'B004EBT5CU', 'title': 'Oscd Ad Adele..."
57293,AFJUWYQH5CADZFZL5FE2WKKY4IJA,"[{'iid': 'B0000CD5FR', 'title': 'Eagles: The V..."
57294,AEV2GTWY2ZIZNNMPMVWQFB3FO7QQ,"[{'iid': 'B0043URV5U', 'title': 'Ultimate Hits..."
57295,AFPHJYF2XL2ID4KRVOZPFA35GKPA,"[{'iid': 'B000089RVX', 'title': 'Fallen', 'cat..."


In [8]:
import sys
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
sys.path.append('../../')
from src.models.vectorDBbuild import VectorDBBuilder
from src.models.hybrid_retrieval import RetrievalEngine
from src.models.newUser import NewUserRetriever

# from sentence_transformers import CrossEncoder  

embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vector_builder = VectorDBBuilder()

# cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')  # Lightweight reranker

/var/folders/1l/fxxxl8p13clcn13ph4b5wcjc0000gn/T/ipykernel_41454/2476412723.py:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


README.md: 0.00B [00:00, ?B/s]

In [9]:
def preprocess_for_augmentation(df):
    for _, row in df.iterrows():
        for rec in row['recommendations']:
            rec['combined_text'] = (
                f"Title: {rec['title']}\n"
                f"Categories: {rec.get('categories', '')}\n"
                f"Description: {rec.get('description', '')}"
            )
            rec['cf_score'] = float(rec.get('score', 0)) 
    return df

In [10]:
df = preprocess_for_augmentation(df)
df

,user_id,recommendations
0,AE4QWKT4VX3DUVNTZM2OHJ2K7YMQ,"[{'iid': 'B000002U82', 'title': 'Dark Side Of ..."
1,AH3EHHDN2SCPSPE5FMS4MHUP5CEA,"[{'iid': 'B000B8QEZG', 'title': 'Confessions o..."
2,AEYOPYIIRMWVY5BLNZKZFZFJFQRA,"[{'iid': 'B00000J2PH', 'title': 'Carole King T..."
3,AHCSCMYFD5WLVXSYWSJGOJD65TMA,"[{'iid': 'B000002UB3', 'title': 'Abbey Road', ..."
4,AECJOIRAVDVSVD6OD5UZKW4PO2KQ,"[{'iid': 'B004EBT5CU', 'title': 'Oscd Ad Adele..."
...,...,...
57292,AGNSSC2S3TL7HZETBWSC6NDILSMA,"[{'iid': 'B004EBT5CU', 'title': 'Oscd Ad Adele..."
57293,AFJUWYQH5CADZFZL5FE2WKKY4IJA,"[{'iid': 'B0000CD5FR', 'title': 'Eagles: The V..."
57294,AEV2GTWY2ZIZNNMPMVWQFB3FO7QQ,"[{'iid': 'B0043URV5U', 'title': 'Ultimate Hits..."
57295,AFPHJYF2XL2ID4KRVOZPFA35GKPA,"[{'iid': 'B000089RVX', 'title': 'Fallen', 'cat..."


In [11]:
items_metadata = []
for _, row in df.iterrows():
    for rec in row['recommendations']:
        items_metadata.append({
            'iid': rec['iid'],
            'title': rec['title'],
            'categories': rec.get('categories', []),
            'description': rec.get('description', ''),
            'cf_score': rec['cf_score']  
        })
items_df = pd.DataFrame(items_metadata).drop_duplicates('iid')

In [12]:
chroma_db = vector_builder.build_from_dataframe(items_df)

In [13]:
retriever = RetrievalEngine(None, embedder)
retriever.chroma_db = chroma_db

In [11]:
# rm
def get_recommendations(user_id, query, cf_weight=0.6, lambda_mmr=0.7):
    """End-to-end pipeline for a user"""
    user_recs = df[df['user_id'] == user_id]['recommendations'].iloc[0]
    
    # Create augmented items 
    augmented = []
    for rec in user_recs:
        chunks = chroma_db.similarity_search(
            query=rec['combined_text'],
            k=3,
            filter={"item_id": rec['iid']}
        )
        augmented.append({
            **rec,
            "vector_context": [chunk.page_content for chunk in chunks],
            "augmented_text": (  
                f"{rec['combined_text']}\n\n" 
                f"Related Context:\n"
                f"{' '.join([chunk.page_content for chunk in chunks])}"
            )
        })
    
    # Hybrid retrieval + MMR
    score = retriever.hybrid_retrieve(augmented, query, cf_weight)
    diversified = retriever.mmr_diversify(score, query, lambda_param=lambda_mmr)
    llm_ranked = llm_rerank(diversified[:20], query, user_id)  # Focus on top 20 for efficiency
    return llm_ranked[:final_k]

In [16]:
# user_id = "AECJOIRAVDVSVD6OD5UZKW4PO2KQ" 
# query = "a song that is relaxing jazz with piano"
# results = get_recommendations(user_id, query)

# # results
# for i, rec in enumerate(results[:10], 1):
#     print(f"{i}. {rec['title']} (Score: {rec['hybrid_score']:.2f})")
#     print(f"   Categories: {rec.get('categories', '')}")
#     print(f"   CF Score: {rec['score']:.2f} | Semantic Score: {rec['semantic_score']:.2f}\n")

In [87]:
def get_recommendations(user_id, query, cf_weight=0.6, lambda_mmr=0.7, debug=False):
    """End-to-end pipeline with debugging options"""
    user_type = "existing" if check_user_type(user_id) else "new"
    
    # 1. Get user recommendations
    user_recs = df[df['user_id'] == user_id]['recommendations'].iloc[0]
    
    # 2. Create augmented items
    augmented = []
    for rec in user_recs:
        chunks = chroma_db.similarity_search(
            query=rec['combined_text'],
            k=3,
            filter={"item_id": rec['iid']}
        )
        augmented.append({
            **rec,
            "vector_context": [chunk.page_content for chunk in chunks],
            "augmented_text": f"{rec['combined_text']}\nRelated Context:\n{' '.join([chunk.page_content for chunk in chunks])}"
        })
    
    # if debug:
    #     print("=== After Augmentation ===")
    #     print(f"Found {len(augmented)} items")
    #     print_sample(augmented[:10])

    # 3. Hybrid retrieval
    scored = retriever.hybrid_retrieve(augmented, query, cf_weight)
    
    if debug:
        print("\n=== After Hybrid Retrieval ===")
        print(f"Top 10 items by hybrid score:")
        for i, item in enumerate(sorted(scored, key=lambda x: x['hybrid_score'], reverse=True)[:10], 1):
            print(f"{i}. {item['title']} (Hybrid: {item['hybrid_score']:.3f})")
            print(f"   Categories: {''.join(item.get('categories', []))}")
            print(f"   Description: {item.get('description', '')[:150]}...\n")

    # 4. MMR Diversification
    diversified = retriever.mmr_diversify(scored, query, lambda_param=lambda_mmr)
    
    if debug:
        print("\n=== After MMR Diversification ===")
        print(f"Top 10 diversified items:")
        for i, item in enumerate(diversified[:10], 1):
            print(f"{i}. {item['title']}")
            print(f"   Hybrid: {item['hybrid_score']:.3f}")
            print(f"   MMR: {item['mmr_score']:.3f}")
            print(f"   Categories: {''.join(item.get('categories', []))}")
            print(f"   Description: {item.get('description', '')[:150]}...\n")
            
    # 5. LLM Reranking
    llm_result = llm_rerank(diversified, query, user_id, user_type)
    
    if debug:
        print("\n=== Final LLM-Ranked Results ===")
        print(f"Top 10 final recommendations:")
        # for i, item in enumerate(llm_ranked[:10], 1):
        #     print(f"{i}. {item['title']}")
        for i, item in enumerate(llm_result['parsed_items'][:10], 1):
            print(f"{i}. {item['title']} (ID: {item['iid']})")
            print(f"   Categories: {''.join(item.get('categories', []))}")
            print(f"   Description: {item.get('description', '')[:150]}...")
            if 'hybrid_score' in item:
                print(f"   Hybrid Score: {item['hybrid_score']:.3f}")
            if 'mmr_score' in item:
                print(f"   MMR Score: {item['mmr_score']:.3f}")
            print("   Reason:")
            if 'llm_reason' in item:
                for reason_line in item['llm_reason'].split('\n'):
                    print(f"     {reason_line}")
            else:
                print("     - No reasoning provided")
            print() 
    
    return llm_result['parsed_items'][:10]

    # return llm_ranked

In [14]:
# def print_sample(items, n=3):
#     """Helper to print sample items"""
#     for i, item in enumerate(items[:n], 1):
#         print(f"{i}. {item['title']}")
#         print(f"   ID: {item['iid']}")
#         print(f"   Hybrid: {item.get('hybrid_score', 0):.2f}")
#         print(f"   Categories: {''.join(item.get('categories', []))}")
#         print(f"   Description: {item.get('description', '')[:100]}...\n")

In [15]:
# # Run with debugging enabled -- rm
# user_id = "AECJOIRAVDVSVD6OD5UZKW4PO2KQ"
# query = "relaxing jazz with piano"
# results = get_recommendations(user_id, query, debug=True)

# # Print final results properly
# print("\n=== Final Output ===")
# for i, rec in enumerate(results[:10], 1):
#     print(f"{i}. {rec['title']}")
#     print(f"   ID: {rec['iid']}")
#     print(f"   Hybrid: {rec.get('hybrid_score', 0):.2f}")
#     print(f"   MMR: {rec.get('mmr_score', 0):.2f}")
#     print(f"   Categories: {', '.join(rec.get('categories', []))}\n")

In [79]:
def generate_prompt(user_id, query, items, user_type):
    
    # Build items list with all relevant metadata
    items_str = "\n".join(
        f"{i}. {item['title']} (ID: {item['iid']}) | "
        f"Categories: {', '.join(item.get('categories', []))} | "
        f"Description: {item.get('description', 'No description')[:100]}... | "
        f"Scores: Hybrid={item.get('hybrid_score', 0):.2f}, MMR={item.get('mmr_score', 0):.2f}"
        for i, item in enumerate(items, 1)
    )
    
    prompt = f"""
    **Music Recommendation Expert System**
    
    === User Context ===
    - User ID: {user_id}
    - Type: {user_type} user
    - Query: "{query}"
    {'- History: Available (prioritize hybrid scores and MMR)' if user_type == 'existing' else '- History: New user (focus on query relevance)'}
    
    === Candidate Items ===
    {items_str}
    
    === Task Instructions ===
    Provide 10 final recommendations with:
    1. Clear ranking from 1-10
    2. Complete item information
    3. Justification for each position
    
    For EXISTING users consider:
    - Hybrid score (70% weight) - Combines personal preferences + query relevance
    - MMR score (30% weight) - Ensures diversity
    - Historical alignment
    
    For NEW users consider:
    - Query relevance (semantic match)
    - General popularity indicators
    - Category diversity
    
    === Required Output Format ===
    === IMPORTANT OUTPUT FORMATTING ===
    Please STRICTLY follow these rules:
    1. Each recommendation MUST start with "[Rank]. [Title] (ID: [exact_item_id])"
    2. MUST include ALL metadata and generate reasoning components for each item:
       • Relevance: Explicitly explain how this matches the query "{query}"
       • Personalization/Popularity: [explanation] 
       • Diversity: How this complements other recommendations
    3. Use ONLY bullet points (•) for reasoning
    4. Maintain EXACTLY the specified indentation
    
    Results example:
    ---- Top 10 Recommendations ----
    Example:
    1. Dark Side of the Moon (ID: B000002UAZ)
    - Categories: Rock, Progressive Rock
    - Description: The Dark Side of the Moon is the eighth studio album...
    - Scores: Hybrid=0.92 | MMR=0.88
    - Reason:
      • Relevance: Perfect match for psychedelic rock query
      • Personalization: Matches user's rock music history
      • Diversity: Brings classic rock representation
    
    [...] (continue numbering to 10)
    
    === Critical Constraints ===
    • MUST include exactly 10 items
    • MUST maintain category diversity (min. 3 distinct categories)
    • For existing users: Hybrid score and MMR are primary factors
    • For new users: Pure query relevance is primary factor
    """
    
    return prompt

In [80]:
# def parse_llm_output(output, items):
#     # parse the LLM's formatted output to extract ranked items
#     ranked = []
#     lines = output.split('\n')
    
#     for line in lines:
#         if re.match(r'^\d+\.', line):  # Lines starting with "1.", "2.", etc.
#             try:
#                 # Extract the position number
#                 rank = int(line.split('.')[0])
#                 # Find the corresponding item
#                 if 1 <= rank <= len(items):
#                     ranked.append(items[rank-1])
#             except (ValueError, IndexError):
#                 continue
#     return ranked[:10] if ranked else items[:10]


def parse_llm_output(output, items, query, user_type):
    results = []
    current_item = None
    lines = output.split('\n')
    
    for line in lines:
        line = line.strip()
        
        # detect new item entry 
        item_match = re.match(r'^(\d+)\.\s+(.+?)\s*\(ID:\s*([^)]+)\)', line)
        if item_match:
            rank = int(item_match.group(1))
            title = item_match.group(2).strip()
            item_id = item_match.group(3).strip()
            
            # Find matching item with fuzzy matching
            matching_items = [
                item for item in items 
                if str(item['iid']).lower() == item_id.lower()
                and title.lower() in item['title'].lower()
            ]
            
            if matching_items:
                current_item = {
                    'item': matching_items[0],
                    'reason': []
                }
                results.append(current_item)
        
        # capture reasoning lines (handles different bullet styles)
        elif current_item and (line.startswith(('•', '-', '*', 'Reason', 'Relevance:', 'Diversity:'))):
            current_item['reason'].append(line)
    
    # format final output with fallback reasoning
    formatted_results = []
    for result in results[:10]:
        formatted_item = result['item'].copy()
        
        # process reasoning lines
        if result['reason']:
            # Group related reasoning points
            formatted_reason = []
            current_category = None
            
            for line in result['reason']:
                if 'Relevance:' in line:
                    current_category = 'Relevance'
                    formatted_reason.append(f"• {line}")
                elif 'Diversity:' in line:
                    current_category = 'Diversity'
                    formatted_reason.append(f"• {line}")
                elif 'Personalization:' in line or 'Popularity:' in line:
                    current_category = 'Personalization' if user_type == 'existing' else 'Popularity'
                    formatted_reason.append(f"• {line}")
                elif current_category:
                    formatted_reason.append(f"   {line.strip()}")
            
            formatted_item['llm_reason'] = '\n'.join(formatted_reason)
        else:
            formatted_item['llm_reason'] = "• No reasoning provided by LLM"
        
        formatted_results.append(formatted_item)
    
    return formatted_results if formatted_results else [
        {**item, 'llm_reason': "• Fallback ranking - no LLM reasoning"} 
        for item in sorted(
            items,
            key=lambda x: x.get('hybrid_score', x.get('semantic_score', 0)),
            reverse=True
        )[:10]
    ]
    
    #     # detect new item entry
    #     if line.strip().startswith(tuple(f"{i}." for i in range(1, 11))):
    #         parts = line.split('(')
    #         if len(parts) > 1:
    #             title = parts[0].split('.', 1)[1].strip()
    #             item_id = parts[1].split(')')[0].strip()
    #             # find matching item
    #             matching_items = [item for item in items 
    #                             if item['iid'] == item_id and item['title'] == title]
    #             if matching_items:
    #                 current_item = {
    #                     'item': matching_items[0],
    #                     'reason': []
    #                 }
    #                 results.append(current_item)
        
    #     # collect reasoning for current item
    #     elif current_item and line.strip().startswith('•'):
    #         current_item['reason'].append(line.strip())
    
    # # format the final output
    # formatted_results = []
    # for result in results[:10]:  
    #     formatted = {
    #         **result['item'],
    #         'llm_reason': '\n'.join(result['reason'])
    #     }
    #     formatted_results.append(formatted)
    
    # return formatted_results if formatted_results else items[:10]

In [81]:
from transformers import pipeline
import torch
import re

def llm_rerank(items, query, user_id, user_type):
    # prompt generation
    prompt = generate_prompt(user_id, query, items, user_type)
    
    # Enhanced LLM setup
    try:
        # handle authentication (only needed once per session)
        if "HUGGINGFACE_TOKEN" in os.environ:
            login(token=os.environ["HUGGINGFACE_TOKEN"])
        else:
            # Alternative: Use a token file or prompt user
            login(token="YOUR_HUGGINGFACE_TOKEN_HERE")  # Replace with your actual token
        
        # initialize the pipeline with error handling
        llm = pipeline(
            "text-generation",
            model="mistralai/Mistral-7B-Instruct-v0.1",
            device_map="auto",
            torch_dtype=torch.float16,
            token=True  # Ensures token is used
        )
        
        # generate output with safety checks
        output = llm(
            prompt,
            max_new_tokens=800,
            do_sample=True,
            temperature=0.7,
            truncation=True
        )[0]['generated_text']

        # return both the raw LLM output and the parsed items
        return {
            'llm_output': output,
            'parsed_items': parse_llm_output(output, items, query, user_type)
        }
        
        # parse the output
        # ranked = []
        # for line in output.split('\n'):
        #     if re.match(r'^\d+\.', line):  # Lines starting with "1.", "2.", etc.
        #         try:
        #             rank = int(line.split('.')[0])
        #             if 1 <= rank <= len(items):
        #                 ranked.append(items[rank-1])
        #         except (ValueError, IndexError):
        #             continue
        
        # return ranked[:10] if ranked else sorted(
        #     items, 
        #     key=lambda x: x.get('hybrid_score', x.get('semantic_score', 0)), 
        #     reverse=True
        # )[:10]
        
    except Exception as e:
        print(f"LLM Reranking Error: {str(e)}")
        # Fallback to non-LLM ranking
        return {
            'llm_output': "LLM Failed - Using default ranking",
            'parsed_items': sorted(
                items,
                key=lambda x: x.get('hybrid_score', x.get('semantic_score', 0)),
                reverse=True
            )[:10]
        }
        
        # return sorted(
        #     items,
        #     key=lambda x: x.get('hybrid_score', x.get('semantic_score', 0)),
        #     reverse=True
        # )[:10]

In [24]:
def check_user_type(user_id):
    return user_id in df['user_id'].values

In [ ]:
# def full_pipeline(user_id, query):
#     is_existing = check_user_type(user_id)
#     user_type = "existing" if is_existing else "new"
#     if check_user_type(user_id):
#         results = get_recommendations(user_id, query)
#     else:
#         retriever = NewUserRetriever(chroma_db, embedder)
#         results = sorted(
#             retriever.retrieve(query),
#             key=lambda x: x['semantic_score'],
#             reverse=True
#         )
    
#     prompt = generate_prompt(user_id, query, results[:50])
#     llm_output = llm(prompt, max_new_tokens=300)[0]['generated_text']
#      try:
#         ranked_indices = [int(x.strip()) for x in llm_output.split(",")]
#         final_results = [results[i-1] for i in ranked_indices if 1 <= i <= len(results)]
#         return final_results[:10]
#     except:
#         return results[:10] 

In [88]:
def full_pipeline(user_id, query, debug):
    """Simplified main pipeline"""
    try:
        if check_user_type(user_id):
            results = get_recommendations(user_id, query, debug=debug)
        else:
            retriever = NewUserRetriever(chroma_db, embedder)
            base_results = sorted(
                retriever.retrieve(query),
                key=lambda x: x['semantic_score'],
                reverse=True
            )
            
            llm_result = llm_rerank(base_results[:50], query, user_id, "new")
            
            if debug:
                print("\n=== Raw LLM Output ===")
                print(llm_result['llm_output'])
                
                print("\n=== Final Recommendations ===")
                for i, item in enumerate(llm_result['parsed_items'][:10], 1):
                    print(f"{i}. {item['title']} (ID: {item['iid']})")
                    print(f"   Categories: {', '.join(item.get('categories', []))}")
                    print(f"   Description: {item.get('description', '')[:100]}...")
                    print(f"   Semantic Score: {item['semantic_score']:.2f}")
                    print("   Reason:")
                    if 'llm_reason' in item:
                        for reason_line in item['llm_reason'].split('\n'):
                            print(f"     {reason_line}")
                    else:
                        print("     • No reasoning provided")
                    print()
            
            return llm_result['parsed_items'][:10]
            
    except Exception as e:
        if debug:
            print(f"\nPipeline Error: {str(e)}")
            print("Falling back to basic semantic search")
        
        # Ultimate fallback
        retriever = NewUserRetriever(chroma_db, embedder)
        return sorted(
            retriever.retrieve(query),
            key=lambda x: x['semantic_score'],
            reverse=True
        )[:10]

In [89]:
# For existing user
results = full_pipeline(
    user_id="AECJOIRAVDVSVD6OD5UZKW4PO2KQ",
    query="relaxing jazz with piano",
    debug=True
)


=== After Hybrid Retrieval ===
Top 10 items by hybrid score:
1. Oscd Ad Adele 21 (Hybrid: 0.479)
   Categories: CDs & Vinyl, Pop, Singer-Songwriters
   Description: 21, is the eagerly awaited sophomore album from British singer-songwriter Adele. It’s the follow up to Adele’s critically acclaimed, Grammy award winn...

2. Partners (Hybrid: 0.446)
   Categories: CDs & Vinyl, Pop, Vocal Pop
   Description: 2014 release from the legendary songbird who duets with award winning, record breaking superstars in their own right including Elvis Presley, Michael ...

3. The Very Best Of Fleetwood Mac (Hybrid: 0.436)
   Categories: CDs & Vinyl, Classic Rock, Album-Oriented Rock (AOR)
   Description: Product Description, In 1975, with Stevie Nicks and Lindsey Buckingham joining Mike Fleetwood, John McVie, and Christine McVie, Fleetwood Mac hit thei...

4. 25 (Hybrid: 0.430)
   Categories: CDs & Vinyl, Pop
   Description: ...

5. Blackstar (Hybrid: 0.423)
   Categories: CDs & Vinyl, Rock
   Descript

In [85]:
# For existing user
results = full_pipeline(
    user_id="AECJOIRAVDVSVD6OD5UZKW4P89IK",
    query="Positive songe with guitar",
    debug=True
)

LLM Reranking Error: name 'login' is not defined

=== Raw LLM Output ===
LLM Failed - Using default ranking

=== Final Recommendations ===
1. Are You Experienced (ID: B000002P5Y)
   Categories: 
   Description: ...
   Semantic Score: 0.45
   Reason:
     • No reasoning provided

2. Are You Experienced (ID: B000002P5Y)
   Categories: 
   Description: ...
   Semantic Score: 0.45
   Reason:
     • No reasoning provided

3. Are You Experienced (ID: B000002P5Y)
   Categories: 
   Description: ...
   Semantic Score: 0.45
   Reason:
     • No reasoning provided

4. Are You Experienced (ID: B000002P5Y)
   Categories: 
   Description: ...
   Semantic Score: 0.45
   Reason:
     • No reasoning provided

5. Reveal (ID: B00005BL29)
   Categories: 
   Description: ...
   Semantic Score: 0.44
   Reason:
     • No reasoning provided

6. Reveal (ID: B00005BL29)
   Categories: 
   Description: ...
   Semantic Score: 0.44
   Reason:
     • No reasoning provided

7. Reveal (ID: B00005BL29)
   Categories:

In [ ]:
# For existing user
results = full_pipeline(
    user_id="AECJOIRAVDVSVD6OD5UZKW4PO2KQ",
    query="relaxing jazz with piano",
    debug=True
)

In [ ]:
# rm
def llm_rerank(items: List[Dict], query: str, user_id: str) -> List[Dict]:
    # Dynamic prompt generation
    prompt = generate_prompt(user_id, query, items)
    
    # Enhanced LLM setup
    llm = pipeline(
        "text-generation",
        model="mistralai/Mistral-7B-Instruct-v0.1",
        device_map="auto",
        torch_dtype=torch.float16
    )
    
    # Get and parse output
    output = llm(
        prompt,
        max_new_tokens=400,
        do_sample=True,
        temperature=0.7
    )[0]['generated_text']
    
    # Parse with error handling
    try:
        ranked = []
        for line in output.split('\n'):
            if re.match(r'^\d+\.', line):  # Lines starting with "1.", "2.", etc.
                rank = int(line.split('.')[0])
                if 1 <= rank <= len(items):
                    ranked.append(items[rank-1])
        return ranked[:10] or items[:10]
    except:
        return sorted(items, key=lambda x: x.get('hybrid_score', x['semantic_score']), reverse=True)[:10]

In [ ]:
def augment_with_metadata(user_id, top_k):
    """retrieve lightfm candidates and augment with metadata from VectorDB"""
    user_recs = df[df['user_id'] == user_id]['recommendations'].iloc[0][:50]
    
    chroma_client = Chroma(
        persist_directory="./content/chroma_db",
        embedding_function=embedding_model
    )
    
    # augment each item with semantic metadata
    augmented_recs = []
    for rec in user_recs:
        # Search VectorDB for metadata chunks related to this item
        metadata_chunks = chroma_client.similarity_search(
            query=rec['title'],  # or use rec['categories']
            k=2,  # Get top 2 relevant metadata chunks
            filter={"asin": rec['iid']}  # Filter by item ID
        )
        
        # Combine original rec with metadata
        augmented_recs.append({
            **rec,
            "metadata": [chunk.page_content for chunk in metadata_chunks],
            "semantic_score": 0.0  # Placeholder for next step
        })
    
    return augmented_recs

In [ ]:
def add_combined_text(row):
    combined_text = f"title_review: {str(row['title_review'])}, text: {str(row['text'])}, title_meta: {str(row['title_meta'])}, features: {str(row['features'])}, description: {str(row['description'])}, details: {str(row['details'])}, categories: {str(row['categories'])}"
    row['combined_text'] = combined_text
    return row

# data = merged_df.map(add_combined_text)
merged_df = merged_df.apply(add_combined_text, axis=1)
# combined_texts = merged_df["combined_text"]
merged_df